# v3.6 New Model: OWL-ViT Open-Vocabulary Detection (Apache-2.0)

OWL-ViT (Open-World Localization ViT) from Google enables zero-shot object detection
given text queries. Uses `owlvit-base-patch32` via HuggingFace Transformers (Apache-2.0).

In [ ]:
from PIL import Image, ImageDraw
import numpy as np

img = Image.new('RGB', (640, 480))
draw = ImageDraw.Draw(img)
draw.rectangle([100, 100, 300, 300], fill=(200, 100, 50))
print('Image size:', img.size)

In [ ]:
try:
    from transformers import OwlViTProcessor, OwlViTForObjectDetection
    import torch

    model_id = 'google/owlvit-base-patch32'
    processor = OwlViTProcessor.from_pretrained(model_id)
    model = OwlViTForObjectDetection.from_pretrained(model_id)

    texts = [['a photo of a cat', 'a photo of a dog', 'a rectangular object']]
    inputs = processor(text=texts, images=img, return_tensors='pt')

    with torch.no_grad():
        outputs = model(**inputs)

    target_sizes = torch.tensor([img.size[::-1]])
    results = processor.post_process_object_detection(outputs, threshold=0.1, target_sizes=target_sizes)[0]

    print(f'Detected {len(results["boxes"])} objects')
    for box, score, label in zip(results['boxes'], results['scores'], results['labels']):
        print(f'  [{texts[0][label]}] score={score:.3f} box={box.tolist()}')
except Exception as e:
    print(f'Skipped (missing deps): {e}')